<a href="https://colab.research.google.com/github/lab-rasool/SIIM/blob/main/notebooks/SIIM_LocalLLMs_Backup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SIIM 2026 Learning Lab — Backup Cloud Instance
### Running Local LLMs Behind Institutional Firewalls (LL4022)

This notebook is a **fallback** for the hands-on labs. If a laptop won't
cooperate during the workshop, run these cells top to bottom to get:

- **Ollama** serving an open model (Lab 1)
- **OpenWebUI** — a private, ChatGPT-style interface, reachable at a public URL (Lab 2)
- the **Lab 3 clinical workflows** (radiology summarization + pathology extraction)

Modeled on [Axenide/Open-WebUI-Colab](https://github.com/Axenide/Open-WebUI-Colab).

---

> ## ⚠️ Read this first — this is the *opposite* of "behind the firewall"
> A Colab instance is a **public cloud machine**. It is the right tool for a
> conference demo with **synthetic data**, and the wrong tool for anything real.
>
> **Use synthetic data only. Never paste real patient data or PHI into this
> notebook or the public OpenWebUI URL it creates.** The whole point of the
> Learning Lab is that the production pattern keeps the model *inside* your
> network — this backup deliberately steps outside it, so treat everything
> here as public.
>
> The public URL is unauthenticated until you create an OpenWebUI account in
> the browser. Shut the runtime down (Runtime → Disconnect and delete runtime)
> when you're done so the tunnel closes.

## Step 0 — Pick your model(s)
`llama3.2` matches Lab 1 (~2 GB, fast). Set `LARGE_MODEL` to also pull a
bigger model for the side-by-side comparison in Lab 3 — leave it blank to skip.

**Tip:** use a GPU runtime (Runtime → Change runtime type → T4 GPU) so the
models run quickly. Ollama uses the GPU automatically when one is present.

In [ ]:
# Workshop model configuration
SMALL_MODEL = "llama3.2"        # Lab 1 default (~2 GB)
LARGE_MODEL = "qwen2.5:7b"     # for the small-vs-large comparison; set to "" to skip

import os
os.environ["SMALL_MODEL"] = SMALL_MODEL
os.environ["LARGE_MODEL"] = LARGE_MODEL

# Ollama performance tuning. Every `ollama serve` we start later inherits this
# environment, so set it once, up front:
#   FLASH_ATTENTION  -> faster, lower-memory attention kernels
#   KV_CACHE_TYPE    -> q8_0 quantizes the KV cache (needs flash attention),
#                       ~halving its memory so longer reports fit on a T4
#   KEEP_ALIVE=-1    -> keep the model resident in VRAM between requests, so
#                       there is no reload pause during the live demo
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"]   = "q8_0"
os.environ["OLLAMA_KEEP_ALIVE"]      = "-1"

print("Will pull:", SMALL_MODEL, "and", LARGE_MODEL or "(no large model)")


## Step 1 — Install Ollama and pull the model(s)
Installs the Ollama runtime, starts it in the background, and pulls your
model(s). On a T4 GPU this takes a couple of minutes.

**Use a GPU runtime before running this:** Runtime → Change runtime type →
**T4 GPU**. Ollama uses the GPU automatically *once it can detect it* — the
cell below installs the detection tools (`pciutils`, `lshw`) so it can. On a
CPU runtime `llama3.2` still works but is slow, and a 7B model is impractical
for a live demo (set `LARGE_MODEL = ""` in Step 0).

In [ ]:
# Ollama's installer ships zstd-compressed archives and detects GPUs via
# lspci/lshw - a fresh Colab runtime has none of these, so install them first.
# Without zstd the install fails; without pciutils/lshw it silently falls back
# to CPU even on a GPU runtime.
!sudo apt-get update -qq
!sudo apt-get install -y -qq zstd pciutils lshw

# Install the Ollama runtime
!curl -fsSL https://ollama.com/install.sh | sh

# Make sure the binary actually landed before we try to run it
import shutil, subprocess, time, os
ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
assert os.path.exists(ollama_bin), (
    'Ollama did not install. Re-run this cell; if it persists, check that '
    'the zstd install above succeeded.'
)

# Is a GPU actually attached? If this errors, you're on a CPU runtime:
# Runtime -> Change runtime type -> T4 GPU, then re-run from this cell.
# (llama3.2 still works on CPU; a 7B model will be very slow.)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU detected - running on CPU.'

# Colab has no systemd, so start the server ourselves in the background.
# This subprocess inherits the OLLAMA_* perf env set in Step 0.
subprocess.Popen([ollama_bin, 'serve'])
time.sleep(8)

# Pull the workshop model(s)
!ollama pull $SMALL_MODEL
if os.environ.get("LARGE_MODEL"):
    get_ipython().system('ollama pull $LARGE_MODEL')

# Confirm what's running locally (Lab 1, step 3)
!curl -s localhost:11434/api/tags | python3 -m json.tool


## Step 2 - Install OpenWebUI
OpenWebUI needs Python 3.11. We use [uv](https://docs.astral.sh/uv/) to do this
in one fast step: uv fetches a managed CPython 3.11 itself (no `apt`) and builds
an isolated virtual environment at `/content/venv`, then installs OpenWebUI into
it much faster than pip. This cell only *installs* - we start the servers a
couple of cells down.

In [ ]:
# Install OpenWebUI with uv. uv creates the Python 3.11 venv at an ABSOLUTE
# path AND fetches a managed CPython 3.11 itself, so the old apt python3.11
# steps are gone. Later cells still expect /content/venv, so the path is
# unchanged.
!pip install -q uv

# Absolute path /content/venv (NOT a relative "venv") so it's found no matter
# what the working directory is when later cells run. --python 3.11 makes uv
# download a managed 3.11 build if the runtime lacks one.
!uv venv --python 3.11 /content/venv
!uv pip install --python /content/venv/bin/python open-webui -q
print("OpenWebUI installed at /content/venv")


## Step 3 — Clone the workshop repo
Brings in the Lab 3 scripts and the **synthetic** clinical datasets.

In [ ]:
import os, glob

# Always use an absolute repo path and only clone if it isn't already there.
# (Re-running this cell will NOT create nested SIIM/SIIM folders.)
REPO_DIR = "/content/SIIM"
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !rm -rf {REPO_DIR}
    !git clone https://github.com/lab-rasool/SIIM.git {REPO_DIR}

# Find where the lab scripts actually live — repo root or a subfolder —
# so this works however the code was pushed to the repo.
hits = glob.glob(os.path.join(REPO_DIR, '**', 'summarize_report.py'), recursive=True)
LAB_DIR = os.path.dirname(hits[0]) if hits else REPO_DIR
os.environ['LAB_DIR'] = LAB_DIR
%cd {LAB_DIR}
print('Lab scripts dir:', LAB_DIR)
!ls -1

## Step 4 - Start OpenWebUI and get a public link
Click this one cell. It starts Ollama, pulls the model, starts the private
ChatGPT-style interface **connected to Ollama**, and prints a public link like
`https://....pinggy.link` via [Pinggy](https://pinggy.io/) (no install - it uses
Colab's built-in `ssh` client). **If no link appears, just click the cell
again** - nothing to type.

Open the link (Pinggy shows a one-time "Enter Site" splash - click through it),
sign up with any email/password (it stays on this runtime), and your pulled
model will be in the dropdown at the top-left. OpenWebUI reaches Ollama
*server-side* over localhost, so the model works the same through the public
link as it does on the box.

> Pinggy's free tunnel lasts 60 minutes and the URL changes each time you start
> a new one - just re-run this cell to get a fresh link.

In [ ]:
# Makes OpenWebUI work end-to-end: starts Ollama, pulls the model, starts
# OpenWebUI pointed at Ollama, and opens a clean public link via Pinggy.
# Click to run. If no link appears, just run it again. No typing required.
import os, re, json, time, shutil, subprocess, urllib.request

def _up(url):
    try: urllib.request.urlopen(url, timeout=3); return True
    except Exception: return False

# 1) Make sure Ollama is running, then pull the model so OpenWebUI has something.
#    (serve inherits the OLLAMA_* perf env set in Step 0.)
ob = shutil.which("ollama") or "/usr/local/bin/ollama"
if not _up("http://localhost:11434/api/tags") and os.path.exists(ob):
    subprocess.Popen([ob, "serve"])
    for _ in range(20):
        if _up("http://localhost:11434/api/tags"): break
        time.sleep(1)
MODEL = os.environ.get("SMALL_MODEL", "llama3.2")
if _up("http://localhost:11434/api/tags"):
    subprocess.run(["ollama", "pull", MODEL])

# 2) Start OpenWebUI, explicitly pointed at the local Ollama. These two env
#    vars are what make your pulled models usable *inside* OpenWebUI: OpenWebUI
#    talks to Ollama server-side over localhost, so it works the same whether
#    you reach the UI locally or through the public Pinggy link.
if not os.path.exists("/content/venv/bin/open-webui"):
    print("\u26a0\ufe0f OpenWebUI isn't installed yet - run Step 2 first, then this cell.")
else:
    if not _up("http://localhost:8081"):
        env = dict(os.environ,
                   OLLAMA_BASE_URL="http://127.0.0.1:11434",
                   ENABLE_OLLAMA_API="true")
        subprocess.Popen(["/content/venv/bin/open-webui", "serve", "--port", "8081"],
                         env=env, stdout=open("/content/openwebui.log","w"),
                         stderr=subprocess.STDOUT)
        print("Starting OpenWebUI\u2026 first boot can take ~30-60s.")
        for _ in range(60):
            if _up("http://localhost:8081"): break
            time.sleep(2)

    # 3) Public link via Pinggy. Open a reverse tunnel for the UI (8081), plus a
    #    LOCAL forward to Pinggy's URL API on 4300 so we can read the real link
    #    as clean JSON instead of scraping the banner. Ollama (11434) stays
    #    private; only the UI is exposed. Free tier: random URL, 60-min sessions.
    get_ipython().system('pkill -f "a.pinggy.io" 2>/dev/null')
    time.sleep(1)
    subprocess.Popen(
        ["ssh", "-p", "443",
         "-o", "StrictHostKeyChecking=no",
         "-o", "UserKnownHostsFile=/dev/null",
         "-o", "ServerAliveInterval=30",
         "-o", "ServerAliveCountMax=3",
         "-R", "0:localhost:8081",
         "-L", "4300:localhost:4300",
         "free@a.pinggy.io"],
        stdin=subprocess.DEVNULL,
        stdout=open("/content/pinggy.log", "w"), stderr=subprocess.STDOUT)

    # The real tunnel URL ends in pinggy.link (or pinggy-free.link). The
    # dashboard link (dashboard.pinggy.io) is shown in the banner but is NOT the
    # tunnel - only accept *.link hosts so we never grab the dashboard by mistake.
    LINK_RE = re.compile(r"https://[a-z0-9.\-]+\.pinggy[a-z0-9.\-]*\.link", re.I)
    url = None
    for _ in range(40):
        # Preferred: Pinggy's local URL API returns the links as clean JSON.
        try:
            data = json.load(urllib.request.urlopen("http://localhost:4300/urls", timeout=3))
            for u in data.get("urls", []):
                if u.startswith("https://"): url = u; break
            if url: break
        except Exception:
            pass
        # Fallback: scrape the SSH log, but only for a real *.link URL.
        try: log = open("/content/pinggy.log").read()
        except FileNotFoundError: log = ""
        m = LINK_RE.findall(log)
        if m: url = m[0]; break
        time.sleep(2)

    if url and _up("http://localhost:8081"):
        print("\n\u2705 OpenWebUI is ready - open this link:\n   " + url)
        print("   (Pinggy shows a one-time 'Enter Site' splash - click through it.)")
        print("   Sign up with any email/password on first visit (stays on this runtime).")
        print("   Your pulled model appears in the dropdown (top-left). Refresh if not.")
    elif url:
        print("\nLink is up but OpenWebUI is still booting - open it in ~30s:\n   " + url)
    else:
        print("\nNo public link yet - just run this cell again.")
        try:
            tail = open("/content/pinggy.log").read().strip().splitlines()[-5:]
            if tail: print("   (last tunnel messages: " + " | ".join(tail) + ")")
        except Exception:
            pass


## Step 5 — Run the Lab 3 clinical workflows
Each cell below makes sure Ollama is running and the model is downloaded
before it runs, so you can just click them — in any order, even if you skipped
a step. All data is synthetic; nothing real goes in.

In [ ]:
# Radiology summarization — makes sure Ollama + the model are ready first.
import os, time, glob, shutil, subprocess, urllib.request

MODEL = os.environ.get("SMALL_MODEL", "llama3.2")
LAB   = os.environ.get("LAB_DIR", "")
if not LAB or not os.path.exists(os.path.join(LAB, "summarize_report.py")):
    hits = glob.glob("/content/SIIM/**/summarize_report.py", recursive=True)
    LAB = os.path.dirname(hits[0]) if hits else "/content/SIIM"

def _up():
    try: urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3); return True
    except Exception: return False

if not _up():
    ob = shutil.which("ollama") or "/usr/local/bin/ollama"
    if os.path.exists(ob):
        subprocess.Popen([ob, "serve"])
        for _ in range(20):
            if _up(): break
            time.sleep(1)

if _up():
    subprocess.run(["ollama", "pull", MODEL])   # instant if already downloaded
    get_ipython().system(f'python3 "{LAB}/summarize_report.py" --model {MODEL} '
                         f'--report "{LAB}/data/radiology/ct_chest_001.txt"')
else:
    print("⚠️ Ollama isn't ready. Please run Step 1 (Install Ollama), then this cell.")

In [ ]:
# Pathology extraction — makes sure Ollama + the model are ready first.
import os, time, glob, shutil, subprocess, urllib.request

MODEL = os.environ.get("SMALL_MODEL", "llama3.2")
LAB   = os.environ.get("LAB_DIR", "")
if not LAB or not os.path.exists(os.path.join(LAB, "extract_pathology.py")):
    hits = glob.glob("/content/SIIM/**/extract_pathology.py", recursive=True)
    LAB = os.path.dirname(hits[0]) if hits else "/content/SIIM"

def _up():
    try: urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3); return True
    except Exception: return False

if not _up():
    ob = shutil.which("ollama") or "/usr/local/bin/ollama"
    if os.path.exists(ob):
        subprocess.Popen([ob, "serve"])
        for _ in range(20):
            if _up(): break
            time.sleep(1)

if _up():
    subprocess.run(["ollama", "pull", MODEL])
    get_ipython().system(f'python3 "{LAB}/extract_pathology.py" --model {MODEL} '
                         f'--report "{LAB}/data/pathology/path_lung_002.txt"')
else:
    print("⚠️ Ollama isn't ready. Please run Step 1 (Install Ollama), then this cell.")

In [ ]:
# Small vs. larger model, side by side. Needs LARGE_MODEL set in Step 0.
import os, time, glob, shutil, subprocess, urllib.request

MODEL = os.environ.get("SMALL_MODEL", "llama3.2")
LAB   = os.environ.get("LAB_DIR", "")
if not LAB or not os.path.exists(os.path.join(LAB, "summarize_report.py")):
    hits = glob.glob("/content/SIIM/**/summarize_report.py", recursive=True)
    LAB = os.path.dirname(hits[0]) if hits else "/content/SIIM"

def _up():
    try: urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3); return True
    except Exception: return False

if not _up():
    ob = shutil.which("ollama") or "/usr/local/bin/ollama"
    if os.path.exists(ob):
        subprocess.Popen([ob, "serve"])
        for _ in range(20):
            if _up(): break
            time.sleep(1)

LARGE = os.environ.get("LARGE_MODEL", "")
if not LARGE:
    print("Set LARGE_MODEL in Step 0 (and run it) to enable the comparison.")
elif _up():
    subprocess.run(["ollama", "pull", MODEL])
    subprocess.run(["ollama", "pull", LARGE])
    get_ipython().system(f'python3 "{LAB}/summarize_report.py" --compare {MODEL} {LARGE} '
                         f'--report "{LAB}/data/radiology/mri_brain_003.txt"')
else:
    print("⚠️ Ollama isn't ready. Please run Step 1 (Install Ollama), then this cell.")